<a href="https://colab.research.google.com/github/Lacenedihia/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lacenedihia/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

# Lane 1: Ranking Signal Analysis.
I chose this line due to the `content_refresh_anonymized.csv` dataset that carries the outcome which is the search position and set of on-page/off-page signal columns such as word count internal links title/meta...

I find it interesting to figure out if there is any groups of signals that move together with the position
a way of finding out what tends to separate high and low position pages

In [ ]:
import os, sys, subprocess
#Imports three standard library modules: os (filesystem/OS operations), sys (interpreter/system info), subprocess (run external commands).
IN_COLAB = "google.colab" in sys.modules
#if google.colab has been imported (which happens automatically in a Colab notebook), this evaluates to True
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    #If the repo folder doesn't already exist, it clones it with git clone --depth 1 (a shallow clone — only the latest commit, no history — to save time/bandwidth).
#Changes the working directory into the cloned repo.
#Installs the repo's Python dependencies from requirements.txt using pip, run quietly (-q). It uses sys.executable (path to the current Python interpreter) rather than just pip, to make sure it installs into the same Python environment Colab is using.
#check=True in each subprocess.run means: if the command fails (non-zero exit code), raise an exception immediately instead of silently continuing.
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
        #Assumes the repo is already cloned locally, but the notebook's kernel might have started in some subfolder.
        #So it walks up the directory tree (os.chdir("..")) repeatedly until it finds a folder containing data/raw (presumably the repo root), or until it hits the filesystem root / (safety stop so it doesn't loop forever).

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("rows, cols:", df.shape)
print("\nall columns:\n", df.columns.tolist())
rank_cols = [c for c in df.columns if any(k in c.lower() for k in ["position", "rank"])]
signal_keywords = ["word_count", "words", "title", "meta", "internal_link", "backlink",
                    "age", "days", "freshness", "h1", "image", "schema", "content_length"]
signal_cols = [c for c in df.columns if any(k in c.lower() for k in signal_keywords)]

print("\ncandidate ranking/position columns:", rank_cols)
print("candidate signal columns:", signal_cols)
assert rank_cols and signal_cols, "This lane needs at least one position column and one signal column — check data-dictionary.md for the real names and adjust the keyword lists above."


rows, cols: (30000, 44)

all columns:
 ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

candidate ranking/position columns: ['avg_position', 'position_tier']
candidate signal columns: ['word_count', 'pageviews_90d', 'engaged_sessions_90d', 'days_with_impressions', 'days_with_s

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision this improves:** when an SEO/content team is deciding what to change on an existing page, which lever to pull first to move its search position such as length , internal links , freshness .

**Who acts on it:** an SEO strategist or content editor working under a fixed time budget; they make one or two targeted edits per page per sprint, not a full rewrite every time.

**Action taken:** the page gets a specific, targeted edit (e.g., add internal links, extend a thin section, refresh the publish date) instead of a generic "go update it" instruction with no guidance on what to fix.

**Cost of a wrong call:** if I tell a strategist "word count is what's holding this page back" and it isn't, they spend real editing time padding a page that won't move and the actual weak signal (say, missing internal links) never gets fixed, so the page's position doesn't change and the team's already limited hours are wasted. If I say a signal doesn't matter when it actually does, the team stops investing in it and leaves real ranking gains on the table. Either mistake is a real opportunity cost, not just a wrong number on a page.

**Why data/ML helps at all:** with ~30,000 pages and a dozen-plus candidate signal columns, "which signal actually correlates with position, and how strongly" isn't something you can eyeball in a spreadsheet — but it's exactly the kind of pattern a straightforward correlation/feature-importance pass can surface across the whole dataset at once, ranked by strength.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
client_cols = [c for c in df.columns if "client" in c.lower() or "domain" in c.lower() or "site" in c.lower()]
print("candidate client/site columns:", client_cols)

if client_cols:
    c = client_cols[0]
    pages_per_client = df.groupby(c).size()
    print(f"\nnumber of distinct clients/sites: {df[c].nunique()}")
    print(f"median pages per client: {pages_per_client.median():.0f}")
    print(f"max pages for a single client: {pages_per_client.max()}")
    print("\nThis is the triage problem in one number: no strategist reads every page for every client by hand every sprint.")



candidate client/site columns: ['client_id']

number of distinct clients/sites: 32
median pages per client: 567
max pages for a single client: 7008

This is the triage problem in one number: no strategist reads every page for every client by hand every sprint.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*
To be sure of my choice I looked at whether any of the candidates signal columns move together with ranking position in the starter sample and how far most pages are from a stong position in the first place


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
numeric_signals = [c for c in signal_cols if pd.api.types.is_numeric_dtype(df[c])]
pos_col = rank_cols[0]
print("using position column:", pos_col)
print("numeric signal columns available:", numeric_signals)

# Number 1 & 2: correlation of each numeric signal with position
# (remember: for most rank fields, a LOWER number = a BETTER position)
if numeric_signals:
    corrs = (
        df[[pos_col] + numeric_signals]
        .corr()[pos_col]
        .drop(pos_col)
        .sort_values(key=lambda s: s.abs(), ascending=False)
    )
    print("\ncorrelation of each signal with", pos_col, ":")
    print(corrs)

# Number 3: how much room for action actually exists
median_pos = df[pos_col].median()
share_beyond_10 = (df[pos_col] > 10).mean()
print(f"\nmedian {pos_col}: {median_pos:.1f}")
print(f"share of pages ranked worse than position 10: {share_beyond_10:.1%}")

using position column: avg_position
numeric signal columns available: ['word_count', 'pageviews_90d', 'engaged_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'engagement_rate']

correlation of each signal with avg_position :
content_age_days          0.158238
age_tier_order            0.151706
word_count                0.123813
days_with_sessions       -0.106046
days_with_impressions     0.078009
days_since_last_update    0.070140
engaged_sessions_90d     -0.054509
engagement_rate          -0.017926
pageviews_90d            -0.017177
Name: avg_position, dtype: float64

median avg_position: 10.8
share of pages ranked worse than position 10: 52.7%


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

1. What is this work can say which signals are observed to move together with ranking position across this sample and how strong and consistent that association is
2. what it can never say is that changing a signal causes a position change for any specific page. This observational cross sectional data not a controlled experiment


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

if len(numeric_signals) >= 2:
    cross_corr = df[numeric_signals].corr().abs()
    np.fill_diagonal(cross_corr.values, 0)
    max_pair = cross_corr.stack().idxmax()
    print(f"most correlated signal pair: {max_pair} at r = {cross_corr.loc[max_pair]:.2f}")
    print("if that number is high, it's a live confound risk for any 'this signal matters most' claim above.")
else:
    print("fewer than 2 numeric signal columns found — revisit the keyword lists against data-dictionary.md.")


most correlated signal pair: ('content_age_days', 'age_tier_order') at r = 0.95
if that number is high, it's a live confound risk for any 'this signal matters most' claim above.


## Self-check

Before you submit, confirm each line honestly:

- [+ ] Every section above is filled — markdown thinking AND the code that backs it
- [+ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+ ] No client names, URLs, or private queries anywhere
- [ +] My claims use careful words: observed, measured, directional, decision-support
- [ +] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.